In [1]:
# exportamos las siguientes variables de entorno
%env AWS_ACCESS_KEY_ID=minio   
%env AWS_SECRET_ACCESS_KEY=minio123 
%env MLFLOW_S3_ENDPOINT_URL=http://localhost:9000

env: AWS_ACCESS_KEY_ID=minio
env: AWS_SECRET_ACCESS_KEY=minio123
env: MLFLOW_S3_ENDPOINT_URL=http://localhost:9000


In [54]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np
import os
import requests
import boto3
import pickle
import io
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Establecemos la URI de tracking de MLflow
mlflow.set_tracking_uri('http://localhost:5001') 

# Definir el experimento en MLflow (se crea si no existe)
experiment_name = "busquedaHiperParamWAPredict"
mlflow.set_experiment(experiment_name)




<Experiment: artifact_location='s3://mlflow/5', creation_time=1746309992353, experiment_id='5', last_update_time=1746309992353, lifecycle_stage='active', name='busquedaHiperParamWAPredict', tags={}>

Busqueda de hiperparametros

In [52]:

# --- S3 Configuration ---
s3_endpoint_url = os.getenv("MLFLOW_S3_ENDPOINT_URL") # Use existing env var
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
bucket_name = 'data' # Make sure this matches your bucket name

# Initialize S3 client
s3_client = boto3.client(
    's3',
    endpoint_url=s3_endpoint_url,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

def load_csv_from_bucket(bucket_name, key):
    # --- Load x_train.csv from S3 ---
    
    print(f"Attempting to load data from: s3://{bucket_name}/{key}")
    try:
        obj = s3_client.get_object(Bucket=bucket_name, Key=key)
        # Use index_col=0 if your CSV saved the DataFrame index as the first column
        csv = pd.read_csv(io.BytesIO(obj['Body'].read()), index_col=0)
        print("loaded successfully from S3.")
        return csv
    except Exception as e:
        print(f"Error loading x_train.csv from S3: {e}")
        raise

X_train = load_csv_from_bucket(bucket_name, 'TransformedData/X_train_transformed.csv')
y_train = load_csv_from_bucket(bucket_name, 'TransformedData/y_train.csv').reset_index()
X_test = load_csv_from_bucket(bucket_name, 'TransformedData/X_test_transformed.csv')
y_test = load_csv_from_bucket(bucket_name, 'TransformedData/y_test.csv').reset_index()


Attempting to load data from: s3://data/TransformedData/X_train_transformed.csv
loaded successfully from S3.
Attempting to load data from: s3://data/TransformedData/y_train.csv
loaded successfully from S3.
Attempting to load data from: s3://data/TransformedData/X_test_transformed.csv
loaded successfully from S3.
Attempting to load data from: s3://data/TransformedData/y_test.csv
loaded successfully from S3.


In [41]:
y_train.reset_index()['y_train']

0         1
1         1
2         0
3         0
4         1
         ..
116363    0
116364    0
116365    0
116366    0
116367    0
Name: y_train, Length: 116368, dtype: int64

In [57]:
with mlflow.start_run():
    mlflow.sklearn.autolog()
    model_wapredict = XGBClassifier(
        use_label_encoder=False, 
        objective='binary:logistic',
        random_state=42
    )
    
    param_grid = {
        'max_depth'        : [6, 8, 10],
        'min_child_weight' : [1, 3, 5],
        'subsample'        : [0.6, 0.8, 1.0],
        'colsample_bytree' : [0.6, 0.8, 1.0]
    }
    
    wapredict_grid = GridSearchCV(
        estimator  = model_wapredict,
        param_grid = param_grid,
        cv         = 5,           # validación cruzada 5 folds
        scoring    = 'f1',  # ver opciones de metricas
        n_jobs     = 12,          # usar todos los núcleos
        verbose    = 1
    )
    
    wapredict_grid.fit(X_train, y_train.reset_index()['y_train'])
    
    print("Mejores parámetros encontrados:", wapredict_grid.best_params_)
    print("Mejor score en CV:",            wapredict_grid.best_score_)
    
    wapredict_results = wapredict_grid.predict(X_train)



Fitting 5 folds for each of 81 candidates, totalling 405 fits


/home/leocenturion/Documents/postgrados/ia/mlops/tp/notebooks/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:47:40] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/leocenturion/Documents/postgrados/ia/mlops/tp/notebooks/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:47:40] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/leocenturion/Documents/postgrados/ia/mlops/tp/notebooks/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:47:40] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/leocenturion/Documents/postgrados/ia/mlops/tp/notebooks/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:47:40] WARNING:

🏃 View run luxuriant-hare-135 at: http://localhost:5001/#/experiments/5/runs/c68634267cdf4ab8a9cab92fa91972d2
🧪 View experiment at: http://localhost:5001/#/experiments/5
🏃 View run brawny-squid-230 at: http://localhost:5001/#/experiments/5/runs/935787650029428494a083e603915c83
🧪 View experiment at: http://localhost:5001/#/experiments/5
🏃 View run delicate-foal-861 at: http://localhost:5001/#/experiments/5/runs/88e459f87ffe4d8180dbe94ff999276b
🧪 View experiment at: http://localhost:5001/#/experiments/5
🏃 View run delicate-croc-986 at: http://localhost:5001/#/experiments/5/runs/b2d6ff06d9d54434bdda1123bec2179b
🧪 View experiment at: http://localhost:5001/#/experiments/5
🏃 View run rambunctious-koi-950 at: http://localhost:5001/#/experiments/5/runs/f7039818a3074f33b0ee6fb47d109d8d
🧪 View experiment at: http://localhost:5001/#/experiments/5
Mejores parámetros encontrados: {'colsample_bytree': 1.0, 'max_depth': 10, 'min_child_weight': 5, 'subsample': 0.8}
Mejor score en CV: 0.645586098772764

array([0, 1, 0, ..., 0, 0, 0], shape=(116368,))

In [55]:
with mlflow.start_run():
    # Se registran los mejores hiperparámetros
    mlflow.log_params(wapredict_grid.best_params_)

    # Se obtiene las predicciones del dataset de evaluación
    y_pred = wapredict_grid.predict(X_test)

    # Se calculan las métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')

    # Y las enviamos a MLFlow
    metrics ={
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall
        }
    mlflow.log_metrics(metrics)

Accuracy: 0.8609239653512993
Precision: 0.8537438463777987
Recall: 0.8609239653512993
🏃 View run orderly-fawn-606 at: http://localhost:5001/#/experiments/5/runs/f67e69b0c3494d8c870dc35eb69f5ba2
🧪 View experiment at: http://localhost:5001/#/experiments/5
